# 天気図のラベリングVSCode でこのノートブックを開き、右上の **「カーネルの選択」** から`venv` の Python を選んでから実行してください。事前に一度だけ:```pip install jupyter ipywidgets ipykernel```VSCode の拡張機能「Jupyter」(Microsoft製)も必要です。

## 1. パスの設定と読み込み

In [ ]:
import sysfrom pathlib import Path# このノートブックの位置からリポジトリのルートを探すREPO = Path.cwd().resolve()while not (REPO / "src" / "labels.py").exists() and REPO != REPO.parent:    REPO = REPO.parentsys.path.insert(0, str(REPO))# 画像とlabels.csvはリポジトリの隣のフォルダに置く想定# (別の場所にある場合はこの1行だけ書き換える)DATA_DIR = REPO.parent / "weather-pattern-classification-data"processed_dir = DATA_DIR / "processed"labels_csv = DATA_DIR / "labels.csv"print("リポジトリ:", REPO)print("画像フォルダ:", processed_dir, "->", "OK" if processed_dir.exists() else "見つかりません")print("labels.csv :", labels_csv, "->", "OK" if labels_csv.exists() else "見つかりません")

## 2. 進捗の確認ラベル済み・未ラベルの枚数を年ごとに表示します。

In [ ]:
import reimport collectionsimport pandas as pddef count_by_year(names):    c = collections.Counter()    for n in names:        m = re.search(r"(\d{10})", str(n))        if m:            c[m.group(1)[:4]] += 1    return cimages = [p.name for p in processed_dir.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg")]labeled = set(pd.read_csv(labels_csv)["filename"]) if labels_csv.exists() else set()img_by_year = count_by_year(images)lab_by_year = count_by_year(n for n in images if n in labeled)print(f"{'年':<8}{'ラベル済み':>10}{'画像total':>10}{'残り':>8}")for year in sorted(img_by_year):    total, done = img_by_year[year], lab_by_year.get(year, 0)    print(f"{year:<8}{done:>10}{total:>10}{total - done:>8}")print(f"\n合計 残り {len(images) - len([n for n in images if n in labeled])} 枚")

## 3. ラベリングチェックボックスで該当するものを選び「決定」で次へ進みます。既にラベル済みの画像は自動的にスキップされます。途中でやめてもその都度 `labels.csv` に保存されているので、セルを再実行すれば続きから再開できます。

In [ ]:
from scripts.label_tool import run_labeling_sessionrun_labeling_session(images_dir=str(processed_dir), labels_csv=str(labels_csv))

## 4. 見直し(任意)新しいラベルを追加したあとに、既存の画像を見直したいときに使います。`month_range` で月を絞れます(例: オホーツク海高気圧なら5〜8月)。

In [ ]:
# from scripts.label_tool import run_review_session## run_review_session(#     images_dir=str(processed_dir),#     labels_csv=str(labels_csv),#     month_range=(5, 8),# )